당신은 'Ko-FreshQA' 벤치마크를 수행하는 정밀한 팩트체크 전문가입니다. 다음 지침을 엄격히 준수하세요.

1. 최신성 우선: 제공된 검색 결과가 있다면, 당신의 내부 지식보다 검색 결과를 우선하여 답변하십시오.
2. 간결성: 부연 설명 없이 질문에 대한 직접적인 정답만 핵심적으로 답변하십시오.
3. 전제 검증 (중요): 질문에 잘못된 사실이나 전제가 포함되어 있다면, 정답을 지어내지 말고 "질문의 전제가 잘못되었습니다"라고 지적한 뒤 올바른 사실을 설명하십시오.
4. 모호성 처리: 현재 시점에서 확정되지 않은 정보라면 "현재 확인할 수 없습니다"라고 답하십시오.

In [1]:
import os
from datetime import datetime

import pandas as pd
from openai import OpenAI

In [ ]:
os.join()

'/data/ephemeral/pro-nlp-finalproject-nlp-13/notebooks/Kim'

In [7]:
ROOT_DIR = "/data/ephemeral/pro-nlp-finalproject-nlp-13/data/"
dir = os.path.join(ROOT_DIR, 'ko-freshqa_2025_dev.csv')
df = pd.read_csv(dir, index_col=False)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              550 non-null    int64  
 1   split           550 non-null    object 
 2   domain          550 non-null    object 
 3   question        550 non-null    object 
 4   effective_year  550 non-null    object 
 5   next_review     283 non-null    object 
 6   false_premise   550 non-null    bool   
 7   num_hops        550 non-null    object 
 8   fact_type       550 non-null    object 
 9   source          550 non-null    object 
 10  answer_0        550 non-null    object 
 11  answer_1        550 non-null    object 
 12  answer_2        134 non-null    object 
 13  answer_3        11 non-null     object 
 14  answer_4        2 non-null      object 
 15  answer_5        0 non-null      float64
 16  answer_6        0 non-null      float64
 17  answer_7        0 non-null      flo

In [8]:
df.head()

,id,split,domain,question,effective_year,next_review,false_premise,num_hops,fact_type,source,answer_0,answer_1,answer_2,answer_3,answer_4,answer_5,answer_6,answer_7,answer_8,answer_9
0,3001,DEV,세계,가장 마지막으로 열린 APEC 정상회의가 몇 번째 정상회의였는지 알려줘.,2024,every_year,False,one-hop,fast-changing,https://www.mofa.go.kr/www/wpge/m_3901/content...,제31차,"2024년에 페루 리마에서 열린 정상회의로, 제31차 정상회의였습니다.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3002,DEV,생활/문화,만화 <원피스>는 한국에 몇 권까지 발간됐어?,2025,occasionally,False,one-hop,fast-changing,https://search.kyobobook.co.kr/search?keyword=...,111권입니다.,2025년 6월 30일에 111권이 발간되었습니다.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3003,DEV,스포츠,배드민턴 선수 안세영의 현재 나이는?,2025,every_year,False,one-hop,fast-changing,https://ko.wikipedia.org/wiki/%EC%95%88%EC%84%...,23세,안세영 선수는 현재 23세입니다.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3004,DEV,생활/문화,네이버 카페 '고양이라서 다행이야'의 가입자 수는 몇 명이야?,2025,every_time,False,one-hop,fast-changing,https://cafe.naver.com/ilovecat,"753,820명","2025년 9월 22일 기준 753,820명입니다.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3005,DEV,날씨,어제 울릉도/독도의 강수량은 얼마였나요?,2025,every_time,False,one-hop,fast-changing,https://www.weatheri.co.kr/bygone/bygone01.php,0.1mm,0.1mm입니다.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df['domain'].value_counts()

domain
경제       77
사회       73
연예       70
IT/과학    65
정치       61
세계       56
스포츠      55
생활/문화    53
날씨       32
UNK       8
Name: count, dtype: int64

In [10]:
df['effective_year'].value_counts()

effective_year
before 2022    268
2025           159
2024            80
2023            23
2022            20
Name: count, dtype: int64

### fact_type
- 시간의 흐름에 따른 답변의 변동 가능성에 따라 분류
- fast-changing: 역사적 사건, 진실과 같이 답변이 거의 변하지 않는 질문
- slow-changing: 답변이 몇 년에 걸쳐 변하는 질문
- never-changing: 답변이 보통 1년 또는 그 이내에 변하는 질문

In [32]:
df['fact_type'].value_counts(dropna=False)

fact_type
never-changing    263
slow-changing     227
fast-changing      60
Name: count, dtype: int64

### num_hops
- 답변을 생성하기 위해 필요한 추론의 개수에 따라 분류
- one hop: 단일 추론
- multi hop: 다중 추론

In [12]:
df['num_hops'].value_counts()

num_hops
one-hop      497
multi-hop     53
Name: count, dtype: int64

In [22]:
multi_hop_question = df[df['num_hops'] == "multi-hop"]['question'].tolist()
multi_hop_answer = df[df['num_hops'] == "multi-hop"]['answer_0'].tolist()

In [29]:
df[df['num_hops'] == "multi-hop"]['fact_type'].value_counts()

fact_type
slow-changing     25
never-changing    21
fast-changing      7
Name: count, dtype: int64

In [24]:
for q,a in zip(multi_hop_question, multi_hop_answer):
    print("==="* 60)
    print(f"[Question]: {q}")
    print(f"[Answer]: {a}")

[Question]: 가장 최근 출시된 아이폰 모델 중 최고 출시가인 모델의 화면 크기는 어떻게 되나요?
[Answer]: 17.4cm (6.9인치)
[Question]: 올해 신형 아이폰의 한국 출시일이 언제야?
[Answer]: 2025년 9월 12일
[Question]: 올해 대한민국에 영향을 준 태풍이 몇 개야?
[Answer]: 없습니다.
[Question]: 현재 진보 진영 차기 대선후보 지지도 2위는 누구인가요?
[Answer]: 정청래
[Question]: 작년 글로벌 제약업계 매출 1위를 기록한 기업을 알려줘. 
[Answer]: 존슨앤드존슨(Johnson & Johnson)
[Question]: 비트코인이 한국에서 법정화폐로 인정된 이유가 뭐야?
[Answer]: 비트코인은 한국에서 법정화폐로 인정되지 않았습니다.
[Question]: LG트윈스 이번 시즌 홍창기 평균 자책점 알려줘
[Answer]: LG트윈스의 홍창기 선수는 투수가 아닌 야수입니다.
[Question]: 청약저축의 월 납입 인정액은 2024년 11월 1일부터 얼마로 상향되나요?
[Answer]: 25만원
[Question]: 이세돌 9단과의 대결로 화제가 된 구글이 만든 바둑 AI 프로그램은 뭐야?
[Answer]: 알파고
[Question]: 날씨와 기후의 차이를 알려줘.
[Answer]: 일반적으로 짧은 시간 시시각각으로 변화하는 날씨 현상을 “기상현상”이라고 부르고 이와 같은 기상현상들의 긴 시간 동안의 평균적인 상태를 “기후”라 합니다.
[Question]: 아모레퍼시픽의 화장품 브랜드 중 한국 전통에서 영감을 받은 브랜드가 뭔지 알려줘.
[Answer]: 설화수
[Question]: 1979년 칸 영화제 황금종려상 수상작의 감독은 누구인가요? 
[Answer]: 프랜시스 포드 코폴라
[Question]: 대한민국의 초대 대통령의 임기기간이 언제부터 언제까지였는지 알려줘.
[Answer]: 1948년 7월 24일~1960년 4월 27일
[Question]: 대한민국에

### Source
- 출처. url로 표기됨
- ko.wikipedia가 전체 데이터셋의 약 50% 차지
- terms.naver.com(네이버 지식백과/사전)
- WebSearch를 할 때, wikipedia.org, yna.co.kr, 네이버 지식백과, 네이버 뉴스 등을 고려하는 전략이 필요함

In [44]:
df['source'].nunique()

541

In [54]:
import pandas as pd
from urllib.parse import urlparse

def extract_host(url: str) -> str:
    if pd.isna(url):
        return "NA"
    s = str(url).strip()
    if not s.startswith(("http://", "https://")):
        s = "https://" + s
    try:
        host = urlparse(s).netloc.lower()
        if host.startswith("www."):
            host = host[4:]
        return host if host else "INVALID"
    except Exception:
        return "INVALID"

df["source_host"] = df["source"].apply(extract_host)

df["source_host"].value_counts()[:33]

source_host
ko.wikipedia.org      278
w.wiki                 23
terms.naver.com        18
yna.co.kr              13
weather.go.kr           7
n.news.naver.com        7
index.go.kr             6
joongang.co.kr          5
chosun.com              5
news.kbs.co.kr          4
data.kma.go.kr          3
newsis.com              3
hani.co.kr              3
sedaily.com             3
mofa.go.kr              3
law.go.kr               3
hankyung.com            3
easylaw.go.kr           3
khan.co.kr              2
news.nate.com           2
news.seoul.go.kr        2
m.dongascience.com      2
asiae.co.kr             2
programs.sbs.co.kr      2
economist.co.kr         2
donga.com               2
oland.kbstar.com        2
bbc.com                 2
news.jtbc.co.kr         2
astro.kasi.re.kr        2
biz.chosun.com          2
kobis.or.kr             2
mohw.go.kr              2
Name: count, dtype: int64

### false premise
- 전제의 유효성 검증
- True: 질문에 포함된 전제 자체가 잘못되어 있음
- False: 질문의 포함된 전제에 문제 없음

- 전략
    - 질문 전제가 사실인지 먼저 판단
    - 틀리면 반박 단답으로 종료
    - 맞으면 정상 단답

In [57]:
df['false_premise'].value_counts(dropna=False)

false_premise
False    452
True      98
Name: count, dtype: int64

In [63]:
pd.set_option('display.max_rows', 100)
df[df['false_premise'] == True][['question','answer_0']]

,question,answer_0
13,한국에서 코로나19로 인한 사망자가 현재 몇 명이야?,코로나19로 인한 사망자 수는 더 이상 집계하지 않습니다.
47,현재 두산 베어스의 감독은 누구야?,현재 두산 베어스는 감독이 없습니다.
49,올해 8월에 서울에는 눈이 얼마나 많이 올까?,2025년 8월 서울에는 눈이 내리지 않았습니다.
50,비트코인이 한국에서 법정화폐로 인정된 이유가 뭐야?,비트코인은 한국에서 법정화폐로 인정되지 않았습니다.
51,최근 발생한 최저시급 인하는 기업에 어떤 영향을 주었나요?,최저시급은 지속적으로 상승하고 있습니다.
52,이강인이 리그 득점왕을 수상한게 몇 년도야?,이강인은 득점왕을 수상한 적이 없습니다.
53,올해 1월에 기온이 30도까지 올라간 날이 언제였어?,"1월은 보통 영하권으로 떨어지기 때문에, 30도 이상 넘어갈 일이 없습니다."
54,"대한민국에서 열대야는 밤 최저 기온이 27도 이상인 날이라고 정의되는데, 이번 여름...",한국의 열대야는 밤 최저 기온이 25도 이상인 밤을 의미합니다.
55,스마트폰 시리즈인 갤럭시 S Pro의 최종 판매량은 몇 대인가요?,갤럭시 S 프로라는 스마트폰 모델은 존재하지 않습니다.
56,종합격투기 선수인 추신수의 현재 나이를 알려줘.,추신수는 야구 선수 출신이며 격투기 선수가 아닙니다.


In [ ]:
# prompt 예시
"""
- 질문의 전제가 사실이 아니거나 성립하지 않으면, 정답은 "전제가 사실이 아닙니다." 형태로 시작한다.
- 정정할 수 있는 단일 사실이 확실할 때만 1문장으로 추가한다.
- 설명/근거/추측/여러 사실 나열 금지.
- 확실하지 않으면 반박 문장 1개로 끝낸다.
"""

'\n- 질문의 전제가 사실이 아니거나 성립하지 않으면, 정답은 "전제가 사실이 아닙니다." 형태로 시작한다.\n- 정정할 수 있는 단일 사실이 확실할 때만 1문장으로 추가한다.\n- 설명/근거/추측/여러 사실 나열 금지.\n- 확실하지 않으면 반박 문장 1개로 끝낸다.\n'

### Answer
1. answer_0
    - 주로 짧은 정답이 많음
    - 75% 가 대부분 21자 이내(단순 글자 수 기준)
1. answer_1
    - answer_0에 비해서 길지만, 그래도 짧음

- 대부분 answer는 짧은 형태
- 대부분 1문장으로 출력되는 형태
- strict을 고려한다면, prompt engineering을 통해 정답을 짧게 반환하도록 설계

In [ ]:
# answer_0
df['answer_0'].apply(lambda x: len(x)).describe()

count    550.000000
mean      15.081818
std       16.884549
min        2.000000
25%        5.000000
50%        8.000000
75%       21.000000
max      143.000000
Name: answer_0, dtype: float64

In [ ]:
# answer_1
df['answer_1'].apply(lambda x: len(x)).describe()

count    550.000000
mean      30.389091
std       16.843906
min        3.000000
25%       20.000000
50%       28.000000
75%       37.750000
max      129.000000
Name: answer_1, dtype: float64

In [41]:
# answer_2
df['answer_2'].dropna().apply(lambda x: len(x)).describe()

count    134.000000
mean      28.813433
std       14.183633
min        2.000000
25%       20.000000
50%       28.000000
75%       35.000000
max       86.000000
Name: answer_2, dtype: float64

In [42]:
# answer_3
df['answer_3'].dropna().apply(lambda x: len(x)).describe()

count    11.000000
mean     28.454545
std      24.586027
min       7.000000
25%      18.000000
50%      23.000000
75%      30.500000
max      98.000000
Name: answer_3, dtype: float64

In [ ]:
# answer 문장 길이 비교
# .을 기준으로 분리
def count_sentences(text):
    if pd.isna(text): return 0
    sentences = [s for s in str(text).split('.') if s.strip()]
    return len(sentences)


df['count_ans0'] = df['answer_0'].apply(count_sentences)
df['count_ans1'] = df['answer_1'].apply(count_sentences)

In [66]:
df['count_ans0'].value_counts()

count_ans0
1    530
2     19
3      1
Name: count, dtype: int64

In [67]:
df['count_ans1'].value_counts()

count_ans1
1    515
2     30
3      3
5      1
6      1
Name: count, dtype: int64

In [68]:
df.groupby('false_premise')[['count_ans0', 'count_ans1']].mean()

,count_ans0,count_ans1
false_premise,,
False,1.035398,1.075221
True,1.051020,1.112245


In [69]:
df['answer_0_len'] = df['answer_0'].apply(lambda x: len(x))
df['answer_1_len'] = df['answer_1'].apply(lambda x: len(x))

In [73]:
df.groupby('false_premise')[['answer_0_len']].describe()

answer_0_len                                                \
                     count       mean        std  min   25%   50%    75%   
false_premise                                                              
False                452.0  11.792035  15.986445  2.0   4.0   7.0  12.00   
True                  98.0  30.255102  11.927021  2.0  23.0  28.0  34.75   

                      
                 max  
false_premise         
False          143.0  
True            74.0

In [74]:
df.groupby('false_premise')[['answer_1_len']].describe()

answer_1_len                                                  \
                     count       mean        std   min    25%   50%    75%   
false_premise                                                                
False                452.0  29.778761  17.264210   3.0  18.00  28.0  37.00   
True                  98.0  33.204082  14.499882  14.0  23.25  28.0  40.75   

                      
                 max  
false_premise         
False          129.0  
True            75.0

In [75]:
df_true = df[df["false_premise"] == True].copy()
df_true.sort_values("answer_0_len", ascending=False)[["question","answer_0","answer_0_len"]].head(20)

,question,answer_0,answer_0_len
548,대한민국에서 대통령을 20년 이상 역임한 사람의 이름을 알려줘.,대한민국에서 20년 이상 대통령을 역임한 사람은 없습니다. 현재 최장 집권한 대통령...,74
283,2024년 9월에 열린 한국-우크라이나 외교장관회담에 참석한 장관은 누구야?,2024년 9월에 있던 외교장관회담은 쿠바와의 회담이었습니다. 또한 우크라이나와의 ...,67
269,사하라 사막에 사는 북극곰은 주로 무엇을 먹어?,"북극곰은 사하라 사막에 살지 않습니다. 북극 지역의 차가운 환경에 적응해 있으며, ...",65
270,2000년에 발생했으며 범국민 금모으기 운동으로 위기를 벗어난 외환 위기를 뭐라고 ...,"범국민 금모으기 운동으로 위기를 벗어난 외환 위기는 IMF 사태라고 부르며, 199...",58
291,1980년에 개봉한 영화 '아바타’의 감독은 누구였지?,"아바타는 1980년에 개봉하지 않았습니다. 2009년에 개봉되었으며, 감독은 제임...",56
295,"2022년 월드컵에서 한국이 우승했을 때, 결승전 상대는 누구였지?",2022년 월드컵에서 한국은 우승하지 않았습니다. 실제로 한국은 16강에서 브라질에...,54
298,2024년 기준 자유한국당의 당대표는 누구입니까?,자유한국당은 2020년 2월 17일 여러 정당들과 미래통합당으로 통합되어 소멸되었...,51
294,2018년에 출간된 '해리 포터와 마법사의 돌’의 저자는 누구야?,해리 포터와 마법사의 돌은 2018년에 출간된 것이 아니라 1997년에 처음 출간되...,51
300,2024년 9월 25일 국무총리실로 걸려온 전화에서 언급한 대통령 서거의 날짜는 언...,대통령 서거를 언급한 전화는 2024년 9월 25일에 인요한 국회의원 사무실로 걸려...,51
319,북극에 위치해 있는 아프리카 대륙은 어떤 기후와 환경을 가지고 있어?,"아프리카 대륙은 북극에 위치한 것이 아니라, 대서양과 인도양 사이에 자리 잡고 있습니다.",49


In [ ]:
### 정답 출력 프롬프트 예시
"""
너는 한국어 QA 어시스턴트다.
다음 규칙을 반드시 지켜라.

[출력 규칙]
- 정답만 한 줄로 출력한다.
- 설명/이유/근거/부연/출처/추측을 쓰지 않는다.
- 문장형(~입니다, ~예요)으로 쓰지 말고, 명사구/숫자/고유명사 형태로만 쓴다.
- 쉼표로 여러 개 나열하지 않는다. (정답이 여러 개면 가장 대표 1개만)
- 날짜/수치/단위가 질문에 필수인 경우에만 포함한다.
"""

### 평가 방식
- upstage solar-pro2-250909로 평가
1. relaxed evaluation
    - 답변이 가진 주요 정보의 정확성에만 초점을 맞춰 평가
    - 환각이나 오래된 정보가 포함되어 있어도, 주요 정보에 영향을 미치지 않으면 정답으로 인정될 수 있음
    - 답변 형식이 잘못된 경우(ex: 다른 언어로 답변)도 허용
1. strict evaluation
    - 주요 정보의 정확성뿐 아니라, 모든 사실이 정확하고 최신이어야 함
    - 사소한 환각이라도 포함되면 정답으로 인정되지 않음
    - "제 지식은 2021년 9월까지입니다…" 같은 오래된 정보 경고 문구는 그 내용이 변경되지 않았음이 명확한 경우에만 정답으로 인정

### 전략
1. 말을 줄인다
    - 추가 설명 -> 추가 사실 주장 -> 오답으로 될 가능성이 높음
    - 단답(명사구/숫자...) 등이 유리함
1. 정확/최신의 정보를 판별
1. 추정을 하지 않는 방식으로 유도